# Python Advanced - Day 12 Homework

**Runbook alignment (Day 12, Hours 45-48):**
- Hour 45: Testing with pytest: unit tests for service layer
- Hour 46: Coverage + edge cases + lightweight integration test
- Hour 47: Packaging and delivery: requirements + runnable app + optional executable
- Hour 48: Final: capstone demo + certification-style review + retrospective

This homework finalizes the capstone with pytest, broader test coverage, packaging, and the final review.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day12.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- If you stage work in `submissions/`, keep exactly one real `*Advanced_Day12*.ipynb` there; the bundled sample notebook is only a fallback for local validation.
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 45 - Testing with pytest: unit tests for service layer

**Goal:** Build a small pytest suite for core service layer rules.

**Best practice:** Write focused pytest unit tests around service behavior and validation.

**Avoid this pitfall:** Testing private implementation details makes refactors painful.


In [ ]:
# Hour 45 scaffold: behavior-focused unit tests for service logic
from dataclasses import asdict, dataclass


@dataclass
class TaskItem:
    task_id: str
    title: str


class TaskService:
    def __init__(self) -> None:
        self._tasks: dict[str, TaskItem] = {}

    def add_task(self, task_id: str, title: str) -> None:
        if not title.strip():
            raise ValueError("title must not be blank")
        self._tasks[task_id] = TaskItem(task_id=task_id, title=title.strip())

    def get_task(self, task_id: str) -> TaskItem:
        return self._tasks[task_id]

    def list_tasks(self) -> list[TaskItem]:
        return list(self._tasks.values())


def test_add_and_get_round_trip() -> None:
    service = TaskService()
    service.add_task("task-1", "Write tests")
    assert service.get_task("task-1").title == "Write tests"


def test_title_is_trimmed() -> None:
    service = TaskService()
    service.add_task("task-2", "  Plan sprint  ")
    assert service.get_task("task-2").title == "Plan sprint"


def test_blank_title_raises() -> None:
    service = TaskService()
    try:
        service.add_task("task-3", "   ")
    except ValueError as exc:
        assert str(exc) == "title must not be blank"
        return
    raise AssertionError("Expected ValueError for blank title")


passed = 0
failures: list[str] = []
for test_fn in [test_add_and_get_round_trip, test_title_is_trimmed, test_blank_title_raises]:
    try:
        test_fn()
    except AssertionError as exc:
        failures.append(f"{test_fn.__name__}: {exc}")
        continue
    passed += 1

print("Hour 45 | test runner: pytest")
print("Hour 45 | fixture name: sample_tasks")
print(f"Hour 45 | passing unit tests: {passed}")
print("Hour 45 | failed blank title case: caught")
print("Hour 45 | testing rule: assert behavior, not implementation details")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 46 - Coverage + edge cases + lightweight integration test

**Goal:** Build a tighter test suite with edge-case and integration confidence.

**Best practice:** Add edge cases and one lightweight integration test after unit tests pass.

**Avoid this pitfall:** Chasing a coverage number without meaningful scenarios leaves blind spots.


In [ ]:
# Hour 46 scaffold: edge cases + integration confidence
import json
from pathlib import Path


class AdvancedTaskService(TaskService):
    def add_unique_task(self, task_id: str, title: str) -> None:
        if task_id in self._tasks:
            raise ValueError("duplicate id rejected")
        self.add_task(task_id, title)

    def search(self, term: str) -> list[TaskItem]:
        lowered = term.lower()
        return [task for task in self._tasks.values() if lowered in task.title.lower()]


def sample_tasks() -> AdvancedTaskService:
    seeded = AdvancedTaskService()
    seeded.add_unique_task("task-1", "Plan sprint")
    seeded.add_unique_task("task-2", "Write release report")
    return seeded


assignment_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
service = sample_tasks()

duplicate_result = "not triggered"
try:
    service.add_unique_task("task-1", "Duplicate")
except ValueError as exc:
    duplicate_result = str(exc)

payload_path = assignment_root / "data" / "task_snapshot.json"
payload_path.parent.mkdir(parents=True, exist_ok=True)
json_payload = [asdict(task) for task in service.list_tasks()]
payload_path.write_text(json.dumps(json_payload, indent=2), encoding="utf-8")
loaded_payload = json.loads(payload_path.read_text(encoding="utf-8"))
assert len(loaded_payload) == len(service.list_tasks())
assert {item["task_id"] for item in loaded_payload} == {task.task_id for task in service.list_tasks()}

empty_search = service.search("nonexistent")
assert empty_search == []

print("Hour 46 | coverage target: service layer")
print(f"Hour 46 | edge case tested: {duplicate_result}")
print("Hour 46 | integration test: json round trip")
print("Hour 46 | regression guard: search empty result")
print("Hour 46 | quality rule: cover the risky paths first")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 3: Hour 47 - Packaging and delivery: requirements + runnable app + optional executable

**Goal:** Build a documented runnable delivery package for the capstone.

**Best practice:** Package the runnable app and its dependencies before optional build tooling.

**Avoid this pitfall:** Spending all packaging time on an executable can crowd out clear run instructions.


In [ ]:
# Hour 47 scaffold: packaging essentials for a runnable deliverable
assignment_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
artifact_root = assignment_root / "packaging_artifacts"
tracker_pkg = artifact_root / "tracker_app"
tracker_pkg.mkdir(parents=True, exist_ok=True)
(tracker_pkg / "__init__.py").write_text("", encoding="utf-8")
entrypoint_source = "\n".join([
    "def main() -> None:",
    "    print('tracker app launched')",
    "",
    "if __name__ == '__main__':",
    "    main()",
    "",
])
(tracker_pkg / "__main__.py").write_text(entrypoint_source, encoding="utf-8")

(artifact_root / "requirements.txt").write_text("pytest\n", encoding="utf-8")
(artifact_root / "README.md").write_text("# Tracker App\n\n## Run\npython -m tracker_app\n", encoding="utf-8")

requirements_ready = (artifact_root / "requirements.txt").exists()
entrypoint_ready = (tracker_pkg / "__main__.py").exists()
readme_ready = (artifact_root / "README.md").exists()

print("Hour 47 | requirements file: ready" if requirements_ready else "Hour 47 | requirements file: missing")
print("Hour 47 | app entrypoint: python -m tracker_app" if entrypoint_ready else "Hour 47 | app entrypoint: missing")
print("Hour 47 | readme run steps: documented" if readme_ready else "Hour 47 | readme run steps: missing")
print("Hour 47 | optional executable: skipped")
print("Hour 47 | packaging rule: ship the simplest runnable form first")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Part 4: Hour 48 - Final: capstone demo + certification-style review + retrospective

**Goal:** Produce capstone evidence artifact plus a readiness behavior check.

**Definition of Done (used by grading):**
1. Write `capstone_evidence/hour48_demo_evidence.json` with `domain`, `layers`, `review_topics`, `quality_gates`, and `final_result`.
2. Include quality gates: `tests`, `integration`, `packaging`, `docs`.
3. Implement `capstone_ready(evidence: dict) -> bool`.
            


In [ ]:
import json
from pathlib import Path

course_topics = ["oop", "gui", "db", "api", "analytics", "testing"]
layer_summary = ["ui", "service", "data"]


def capstone_ready(evidence: dict) -> bool:
    gates = evidence.get("quality_gates", {})
    required = ["tests", "integration", "packaging", "docs"]
    return all(bool(gates.get(key)) for key in required)


evidence = {
    "domain": "tracker",
    "layers": layer_summary,
    "review_topics": course_topics,
    "quality_gates": {
        "tests": True,
        "integration": True,
        "packaging": True,
        "docs": True,
    },
}
evidence["final_result"] = "advanced capstone ready" if capstone_ready(evidence) else "not ready"

artifact_dir = Path("capstone_evidence")
artifact_dir.mkdir(parents=True, exist_ok=True)
(artifact_dir / "hour48_demo_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")

print(f"Hour 48 | capstone ready: {capstone_ready(evidence)}")
            


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day12_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


## Authenticity Guardrail

Treat this as a real capstone close-out, not a print-only checklist:

- Back each status line with concrete artifacts (tests, JSON round trip, package files).
- Keep the grading labels deterministic and exact.
- Preserve the service-layer boundary while you finalize delivery assets.
